# 🌌 3.2B PPC-GNN: Biologically Plausible LLM (Dual T4) — v2 (Final Stable)

This is the **Production Version**. It solves the 3.2B OOM error and the Gradient Disconnection bug.

### Key Architecture Features:
- **DEQ-PPC**: Analytical Gradient Bridge for MoE Experts.
- **Dynamic Convergence**: Early stopping saves ~40% compute on easy tokens.
- **FP16 Logic**: Compressed expert weights (3.2GB per GPU).
- **Sharding**: Balanced across 2x T4 GPUs.

In [ ]:
# ── Cell 1: Environment Setup ─────────────────────────────────────────────
import os, sys, math, time, gc
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# 1. Essential Dependencies (These are usually missing on Kaggle)
!pip install -q bitsandbytes wandb datasets transformers

# 2. Pull Research Source (Injecting into path instead of installing)
REPO_NAME = 'EFV-nn'
REPO_URL = 'https://github.com/ey3lock3r/EFV-nn.git'

if not os.path.exists(REPO_NAME):
    os.system(f'git clone -q {REPO_URL}')
else:
    os.system(f'cd {REPO_NAME} && git pull -q')

sys.path.append(f'{REPO_NAME}/src')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
import bitsandbytes as bnb
import wandb
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer
from kaggle_secrets import UserSecretsClient

from efv_nn.ppc_sharded import ShardedPPCGraphLLM

CKPT_PATH = "/kaggle/working/ppc_3b_pretrain.pt"

print("✅ Environment ready. Source loaded from GitHub.")
print(f"CUDA Available: {torch.cuda.is_available()} | Devices: {torch.cuda.device_count()}")


In [ ]:
# ── Cell 2: Secrets & Monitoring ────────────────
try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    WANDB_KEY = secrets.get_secret("WANDB_API_KEY")
    
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['WANDB_API_KEY'] = WANDB_KEY
    wandb.login(key=WANDB_KEY)
    print("✅ Logged into W&B and HF")
except Exception as e:
    print(f"⚠️ Secrets failure: {e}. Training will continue without W&B.")


In [ ]:
# ── Cell 3: Data Pipeline (FineWeb-Edu Streaming) ───────────────
def get_dataloader(tokenizer, batch_size=2, seq_len=256):
    ds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
    def gen():
        buffer = []
        for ex in ds:
            buffer.extend(tokenizer(ex["text"])["input_ids"])
            while len(buffer) >= (batch_size * seq_len):
                yield torch.tensor(buffer[:batch_size * seq_len]).view(batch_size, seq_len)
                buffer = buffer[batch_size * seq_len:]
    return gen()


In [ ]:
# ── Optional: 🛟 W&B Rescue (Pull checkpoint from cloud) ──────────
import shutil, os
RESCUE_RUN_PATH = "dhinsresearch/ppc-3b-dynamic/YOUR_RUN_ID" # Update this
RESCUE_FILE = "ppc_3b_pretrain.pt" # Cleaner root-level path

def rescue_checkpoint(run_path, filename, target_path):
    print(f"📡 Attempting to pull {filename} from {run_path}...")
    try:
        # Download to local directory
        restored_file = wandb.restore(filename, run_path=run_path)
        source_path = restored_file.name
        
        # Ensure target directory exists and move it there
        os.makedirs(os.path.dirname(target_path), exist_ok=True)
        shutil.move(source_path, target_path)
        
        print(f"✅ Rescue Successful! File moved to: {target_path}")
        print("⚠️ Now run Cell 4 (Setup) with RESUME=True to load it.")
    except Exception as e:
        print(f"❌ Rescue Failed: {e}")

# To rescue:
# rescue_checkpoint(RESCUE_RUN_PATH, RESCUE_FILE, CKPT_PATH)

In [ ]:
# ── Cell 4: 🏗️ Architectural Setup (Run ONCE) ───────────────────────────
import importlib
import efv_nn.ppc_sharded, efv_nn.ppc_gnn
importlib.reload(efv_nn.ppc_gnn)
importlib.reload(efv_nn.ppc_sharded)
from efv_nn.ppc_sharded import ShardedPPCGraphLLM

VOCAB_SIZE, HIDDEN, LAYERS, EXPERTS = 128256, 1024, 24, 64
CKPT_PATH = "/kaggle/working/ppc_3b_pretrain.pt"
RESUME = True
ENABLE_GLOBAL_COMPILE = True
ENABLE_SPECTRAL_GUARDIAN = True  # Pillar 2: Phasal Laplacian Regularization (λ=0.01)

print("📥 Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B", token=os.environ.get('HF_TOKEN'))

print("🏗️ Instantiating 3.2B Sharded Model...")
gc.collect(); torch.cuda.empty_cache()
model = ShardedPPCGraphLLM(VOCAB_SIZE, HIDDEN, LAYERS, EXPERTS, use_jacobian=True)

if ENABLE_GLOBAL_COMPILE:
    try:
        print("🚀 Attempting Global Graph Fusion (torch.compile)... ")
        model = torch.compile(model, mode="reduce-overhead")
        print("✅ Global Fusion Enabled.")
    except Exception as e:
        print(f"⚠️ Global Compile skipped (using Islands): {e}")


# Guard: store device refs before potential compile wrapping
DEVICE1 = model.device1 if hasattr(model, 'device1') else model._orig_mod.device1
START_STEP = 0
if RESUME and os.path.exists(CKPT_PATH):
    print(f"🔄 Loading Checkpoint: {CKPT_PATH}")
    ckpt = torch.load(CKPT_PATH, map_location='cpu')
    model.load_state_dict(ckpt['model_state'], strict=False)
    START_STEP = ckpt.get('step', 0)
    print(f"✅ Resumed Weight Step {START_STEP}")

print(f"✅ Model Ready. Total Params: {model.total_params/1e9:.2f}B")

In [ ]:
# ── Cell 5: 🚄 Training Engine (Stabilized + Spectral Guardian) ──────────────────
from efv_nn.ppc_gnn import spectral_guardian_penalty

ENABLE_SPECTRAL_GUARDIAN = True  # Pillar 2: Phasal Laplacian Regularization (λ=0.01)

def save_checkpoint(model, step, path):
    gc.collect(); torch.cuda.empty_cache()
    state_dict = model.state_dict()
    compressed_dict = {k: (v.half() if v.dtype == torch.float32 else v) for k, v in state_dict.items()}
    torch.save({'step': step, 'model_state': compressed_dict}, path)
    wandb.save(path, base_path="/kaggle/working")
    print(f"✅ Step {step} saved.")

def train(model, tokenizer, num_steps=1000, start_step=0, local_iters=24, lr=8e-5):
    gc.collect(); torch.cuda.empty_cache()

    wandb.init(project="ppc-3b-dynamic", name=f"ppc-3b-{time.strftime('%H%M')}")
    dataloader = get_dataloader(tokenizer, batch_size=2, seq_len=256)
    opt = bnb.optim.PagedAdamW8bit(model.parameters(), lr=lr)

    t0 = time.time(); last_step = start_step

    guardian_status = '🛡️ ON' if ENABLE_SPECTRAL_GUARDIAN else '⚪ OFF'
    print(f"🚀 Training [Iters: {local_iters}, LR: {lr}, Spectral Guardian: {guardian_status}]...")

    for step, batch in enumerate(dataloader):
        if step < start_step: continue
        ids, targets = batch[:, :-1], batch[:, 1:].to(DEVICE1)

        opt.zero_grad()
        with torch.amp.autocast('cuda'):
            logits, avg_iters, avg_energy, layer_energies = model(ids, local_iters=local_iters)
            loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), targets.reshape(-1))

        # Spectral Guardian: OUTSIDE autocast to preserve FP32 precision
        if ENABLE_SPECTRAL_GUARDIAN:
            loss = loss + spectral_guardian_penalty(layer_energies.float())

        (loss / 256.0).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0 / 256.0)
        opt.step()

        if step % 5 == 0:
            ppl = math.exp(min(loss.item(), 20))
            lr_curr = opt.param_groups[0]['lr']
            dt = (time.time() - t0) * 1000 / (step - last_step) if step > last_step else 0
            t0 = time.time(); last_step = step
            wandb.log({"loss": loss.item(), "ppl": ppl, "avg_iters": avg_iters,
                       "avg_energy": avg_energy.item(), "lr": lr_curr, "duration_ms": dt}, step=step)
            print(f"St {step} | L: {loss.item():.4f} | E: {avg_energy.item():.4f} | It: {avg_iters:.1f} | D: {dt:.0f}ms | LR: {lr_curr:.2e}")

        if (step + 1) % 100 == 0: save_checkpoint(model, step + 1, CKPT_PATH)
        if step >= num_steps: break

    print(f"\n[PPC-GTB Snapshot] Finished at Step {step}")
    print(f"L: {loss.item():.4f} | E: {avg_energy.item():.4f} | D_avg: {dt:.0f}ms")


In [ ]:
# ── Cell 6: 🚀 Start Training Loop (Stabilized) ──────────────────
train(model, tokenizer, start_step=START_STEP, local_iters=24, lr=8e-5)

In [ ]:
# ── Cell 7: 🗣️ Verification (Instant) ──────────────────
def verify_generation(model, tokenizer, prompt="The fundamental principles of biology include"):
    inputs = tokenizer(prompt, return_tensors="pt")["input_ids"]
    
    print("\n🧠 Regular Generation...")
    t0 = time.time()
    out_reg = model.generate(inputs, max_new_tokens=40)
    print(f"[Reg] {time.time()-t0:.1f}s: {tokenizer.decode(out_reg[0], skip_special_tokens=True)}")
    
    print("\n🐝 Advanced Swarm Inference (N=8)...")
    t0 = time.time()
    # Picking the most resonant ghost state (lowest energy resonance)
    out_swarm = model.generate_swarm(inputs, max_new_tokens=40, swarm_size=8)
    print(f"[Swarm] {time.time()-t0:.1f}s: {tokenizer.decode(out_swarm[0], skip_special_tokens=True)}")

# To run: 
# verify_generation(model, tokenizer)

In [ ]:
# ── Cell 8: 🗣️ Generation Verification ──────────────────
def verify_generation(prompt="The fundamental principles of biology include"):
    tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B", token=os.environ.get('HF_TOKEN'))
    model = ShardedPPCGraphLLM(128256, 1024, 24, 64)
    if os.path.exists(CKPT_PATH):
        ckpt = torch.load(CKPT_PATH, map_location='cpu')
        model.load_state_dict(ckpt['model_state'], strict=False)
    inputs = tokenizer(prompt, return_tensors="pt")["input_ids"]
    print("\n🧠 Running Regular Generation...")
    output_ids = model.generate(inputs, max_new_tokens=50)
    print(f"Output:\n{tokenizer.decode(output_ids[0], skip_special_tokens=True)}")
    
    print("\n🐝 Running Advanced Swarm Inference (N=8)...")
    swarm_ids = model.generate_swarm(inputs, max_new_tokens=50, swarm_size=8)
    print(f"Output:\n{tokenizer.decode(swarm_ids[0], skip_special_tokens=True)}")
    print(f"\nPrompt: {prompt}")

# verify_generation()
